In [8]:
import requests
import base64
import pandas as pd
import plotly.express as px
import jalali_pandas

In [9]:
access_token = "bbb0033f20b4425b90dcf07775c68252"
# Generate Header: "Basic " + Base64(token + ":")
auth_str = f"{access_token}:"
encoded_token = base64.b64encode(auth_str.encode('ascii')).decode('ascii')
headers = {
    "Authorization": f"Basic {encoded_token}",
    "Accept": "application/json"
}

In [11]:
def retrative_read_data(url, params={}):
    page_size = 100  # تعداد رکورد در هر درخواست (بر اساس مثال شما)
    current_skip = 0
    params['_count'] = page_size
    params['_skip'] = current_skip
    data = pd.DataFrame()
    while True:
        params['_skip'] = current_skip
        # print(f"در حال دریافت رکوردهای {current_skip} تا {current_skip + page_size}...")
        df = (pd.DataFrame(requests.get(url, headers=headers, params=params).json()['data']))
        data = pd.concat([data, df])
        
        if len(df) < page_size:
            break
        
        current_skip += page_size
    
    if 'date_time' in data.columns:
        try:
            data['j_date_time'] = data['date_time'].str[:-6].jalali.parse_jalali("%Y%m%d").apply(lambda x: str(x).split(' ')[0])
            data['datetime'] = data['date_time'].str[:-6].jalali.parse_jalali("%Y%m%d").jalali.to_gregorian() 
        except:
            pass 

    return data

In [ ]:
code = {
    "میلگرد": "SteelRebar  ",
    "گندله سنگ آهن": "CD1IOP0001  ",
    "شمش روی": "D1ZNI0001  ",
    "شمش سرب": "LeadIngot   ",
    "گواهی سپرده شمش نقره": "SilverBar   ",
    "مس کاتد": "CopperCthd  ",
    "گواهی سپرده شمش طلا": "GoldBar"
}

In [15]:
retrative_read_data(url = "https://v1.db.api.mabnadp.com/exchange/intradaytrades", params={'id': '1'})

""
